# Prompt Engineering Techniques

## Starter

In [ ]:
# Import
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [ ]:
# Helper
load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

print("Client ready:", model)

# Helper
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

### Prompt Evaluation

In [ ]:
class PrmptEvaluator:
    
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks
        
    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}])}", template_string)
        
        result = template_string
        for placholder in placeholders:
            if placholder in variables:
                result = result.replace(
                    "{" + placholder + "}", str(variables[placholder])
                )
        return result.replace("{{", "{").replace("}}", "}")
    
    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        """Generate a list of unique ideas for test cases based on the task description"""
        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes thst task:
        
        <task_description>
            {task_description}
        </task_description>
        
        The prompt will receive the following inputs
        <prompt_inputs>
            {prompt_inputs_spec}
        </prompt_inputs>
        
        Each idea should represent a distinct scenario or example that tests different aspects of the task.
        
        Output Format:
        Provide your response as a structured JSON array where each item is a brief description of the idea.
        
        Example:
        ```json
        [
            "Testing with technical computer science terminology",
            "Testing with medical research findings",
            "Testing with complex mathematical concepts",
            ...
        ]
        ```
        
        Ensure each idea is:
        - Clearly distinct from the others
        - Relevant to the task description
        - Specific enough to guide generation of a full test case
        - Quick to solve without requiring extensive computation or multi-step processing
        - Solvable with no more than 400 tokens of output
        
        Remember, only generate {num_cases} unique ideas
        """
        
        system_prompt = "You are a tester scenario designer specialized in creative diverse, unique testing scenarios."
        
        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'"{key}": str # {val},'
        
        rendered_prompt = self.render(
            dedent(prompt),{
                "task_description": task_description,
                "num_cases": num_cases,
                "prompt_inputs": example_prompt_inputs,
            },
        )
        

    